# آموزشِ صدا برای موتور محتوا

این نوت‌بوک **یک بار برای هر صدا** اجرا می‌شود. خروجی‌اش دو فایل است که
در درایو ذخیره می‌شوند و از آن به بعد موتور از همان‌ها استفاده می‌کند.

## کاری که باید بکنید

۱. از منوی بالا: `Runtime` ← `Change runtime type` ← **`T4 GPU`**
۲. سلولِ «تنظیمات» را نگاه کنید (معمولاً نیازی به تغییر ندارد)
۳. `Runtime` ← `Run all`

بعدش کاری ندارید. حدود یک ساعت طول می‌کشد. اگر مرورگر را ببندید
اجرا متوقف می‌شود، پس تب را باز بگذارید.

---

منطقِ همهٔ قدم‌ها در مخزن است (`rvcpipe.py` و `dsprep.py`) و همین نوت‌بوک
آن‌ها را از آنجا می‌گیرد — پس یک نسخه بیشتر وجود ندارد و این فایل هرگز
از کدِ اصلی عقب نمی‌افتد. همان چیزی که اینجا دیتاست را می‌سازد، در
آزمایشگاه هم ساخته و با گوش داوری شده است.

## ۱ — کارتِ گرافیک

اگر اینجا خطا داد یعنی GPU روشن نیست. برگردید بالا و از `Runtime`
انتخابش کنید. بی آن، آموزش به‌جای یک ساعت یک روز طول می‌کشد.

In [ ]:
import subprocess, sys, os, shutil, json

g = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total',
                    '--format=csv,noheader'], capture_output=True)
name = (g.stdout or b'').decode().strip()
if g.returncode != 0 or not name:
    raise SystemExit('GPU روشن نیست: Runtime ← Change runtime type ← T4 GPU')
print('کارت:', name)

## ۲ — تنظیمات

برای صدای رضوی همین‌طور که هست درست است. برای صدای **دیگری**:
`VOICE` را عوض کنید و شناسه‌های فایل‌های آن صدا را در `DRIVE_IDS`
بگذارید (شناسه همان رشتهٔ میانِ نشانیِ فایل در درایو است).

In [ ]:
VOICE = 'razavi'          # نامِ صدا؛ نامِ فایل‌های خروجی از این می‌آید

DRIVE_IDS = [            # ضبط‌های همان گوینده در درایو
    '1YRI2p7Qv3hh2dcNPMZDmbNUel0XCYWKX',
    '1cBUasKKB2Q5JjLfpZfyjBC7ZNo72KAiD',
    '1izlhA9PRU0VWcmL-Gw7lFaW2LJ3nLUKv',
    '1QdJzUi8sk5LhuUqjHeCi9Kb4UgRYq4P5',
]

SR      = '40k'   # نرخِ نمونه. ۴۰k برای روایت بس است
EPOCHS  = 150     # کمتر: ناقص. خیلی بیشتر: صدا خشک می‌شود
BATCH   = 8       # روی T4 جا می‌شود
SAVE_TO = '/content/drive/MyDrive/voice-models'   # مقصد در درایو

## ۳ — کد

مخزنِ RVC (پروانهٔ MIT) و دو فایلِ منطقِ خودمان از گیت‌هاب.

In [ ]:
os.chdir('/content')
if not os.path.isdir('/content/rvc'):
    subprocess.run(['git', 'clone', '--depth', '1', '-q',
                    'https://github.com/RVC-Project/'
                    'Retrieval-based-Voice-Conversion-WebUI',
                    '/content/rvc'], check=True)

# منطق از خودِ مخزن می‌آید، نه از نسخه‌ای داخلِ این فایل. پس هرچه در
# آزمایشگاه اصلاح شود، همین‌جا هم اصلاح شده است.
RAW = 'https://raw.githubusercontent.com/mahdighandi1989/Content-Engine/main/tools'
for f in ('rvcpipe.py', 'dsprep.py'):
    subprocess.run(['curl', '-sSLf', RAW + '/' + f,
                    '-o', '/content/' + f], check=True)

sys.path.insert(0, '/content')
import rvcpipe as P, dsprep as D
ROOT = '/content/rvc'
print('قدم‌ها:', [n for n, _ in P.steps(VOICE, '/x', ROOT, sr=SR)])

## ۴ — وابستگی‌ها

فهرست از `rvcpipe.py` می‌آید — همان که در آزمایشگاه هم نصب می‌شود.
`torch` عمداً در آن نیست: Colab نسخهٔ جفت‌شده با CUDAِ خودش را دارد و
نصبِ دوباره‌اش فقط می‌شکندش.

In [ ]:
# دو فهرست، هر دو از خودِ مخزن: یکی برای آموزشِ RVC، یکی برای
# فیلترِ موسیقی و تشخیصِ گوینده. اینجا نوشته نمی‌شوند تا با آنچه
# آزمایشگاه نصب می‌کند فرق نکنند.
deps = P.TRAIN_DEPS + D.DS_DEPS + ['gdown']
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q']
               + deps, check=True)
print('نصب شد:', len(deps), 'بسته')

## ۵ — وزن‌های پایه

حدود ۵۵۰ مگابایت. پروانهٔ همه سنجیده شده: کدِ RVC ‏MIT · وزن‌ها ‏MIT ·
ContentVec ‏MIT · RMVPE ‏Apache-2.0.

In [ ]:
os.chdir(ROOT)
hf = shutil.which('hf') or shutil.which('huggingface-cli')
for cmd in P.assetCmds_(py=sys.executable, sr=SR, hf=hf or 'hf'):
    if cmd[:3] == [sys.executable, '-m', 'pip']:
        continue          # همین بالا با سقفِ درست نصب شد
    if cmd[0] in ('hf', 'huggingface-cli'):
        cmd[0] = hf or cmd[0]
    r = subprocess.run(cmd)
    if r.returncode:
        raise SystemExit('دانلود ناموفق: ' + ' '.join(cmd[:4]))
print('وزن‌ها آمدند')

## ۶ — گرفتنِ ضبط‌ها

In [ ]:
import gdown
RAWDIR = '/content/raw'
os.makedirs(RAWDIR, exist_ok=True)
srcs = []
for i, fid in enumerate(DRIVE_IDS):
    dst = os.path.join(RAWDIR, 'in%d' % (i + 1))
    gdown.download(id=fid, output=dst, quiet=True)
    if os.path.exists(dst) and os.path.getsize(dst) > 100000:
        srcs.append(dst)
    else:
        print('نیامد (دسترسی؟):', fid)
print('%d از %d ضبط آمد' % (len(srcs), len(DRIVE_IDS)))
if not srcs:
    raise SystemExit('هیچ فایلی نیامد — دسترسیِ اشتراکِ فایل‌ها را ببینید')

## ۷ — جداکردنِ موسیقی و ساختِ دیتاست

تیزر، میان‌برنامه و هر جایی که موسیقی زیرِ روایت باشد کنار گذاشته
می‌شود. موسیقی در دادهٔ آموزش سم است: مدل رنگِ صدا را از هرچه در فایل
باشد یاد می‌گیرد.

دو فایلِ `SAMPLE-*.wav` هم ساخته می‌شود — اگر خواستید با گوش بسنجید
چه نگه داشته و چه دور ریخته.

In [ ]:
DS = '/content/dataset'
os.makedirs(DS, exist_ok=True)
segs, rep = D.buildDataset_(srcs, DS, sampleDir='/content')
print(json.dumps({k: v for k, v in rep.items() if k != 'files'},
                 ensure_ascii=False, indent=1))
for row in rep['files']:
    print('%-22s %6.1f ثانیه → %3d تکه' % (row['file'][:22], row['seconds'],
                                          row.get('segments', 0)))
if not segs:
    raise SystemExit('هیچ تکه‌ای نماند — ضبط‌ها را ببینید')
print('\nمجموع: %d تکه، %.1f دقیقه' % (len(segs), rep['kept_seconds'] / 60))

## ۸ — آموزش

پنج قدم، پشتِ هم. طولانی‌ترینش خودِ آموزش است. اگر قدمی خطا داد،
همین‌جا می‌ایستد و علتش را می‌نویسد — جلوتر نمی‌رود تا آخرش معلوم شود
چیزی ساخته نشده.

In [ ]:
import time
P.preLog_(ROOT, VOICE)
env = P.env(ROOT)
steps = P.steps(VOICE, DS, ROOT, sr=SR, f0method='rmvpe', epochs=EPOCHS,
                save_every=max(1, EPOCHS // 5), version='v2', gpus='0',
                n_p=2, batch=BATCH, py=sys.executable)
for nm, cmd in steps:
    if nm == 'train':
        info = P.preTrain_(ROOT, VOICE, sr=SR, version='v2')
        print('فهرستِ آموزش:', info)
        if not info['from_dataset']:
            raise SystemExit('فهرست خالی است — استخراج چیزی نساخت')
    t0 = time.time()
    print('\n=== %s ===' % nm, flush=True)
    r = subprocess.run(cmd, cwd=ROOT, env=env)
    print('%s: %ds' % (nm, time.time() - t0))
    if r.returncode:
        raise SystemExit('قدمِ «%s» شکست خورد (کد %d)' % (nm, r.returncode))

## ۹ — ذخیره در درایو

اجازهٔ دسترسی به درایو را می‌پرسد. دو فایل ذخیره می‌شود: مدل و ایندکس.
**همین دو تا** چیزی است که از این نوت‌بوک لازم داریم.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
o = P.outputs(VOICE, ROOT)
if not os.path.exists(o['model']):
    raise SystemExit('آموزش تمام شد ولی مدلی ساخته نشد: ' + o['model'])
os.makedirs(SAVE_TO, exist_ok=True)
saved = [shutil.copy(o['model'], SAVE_TO)]
for f in sorted(os.listdir(o['index_dir'])):
    if f.endswith('.index'):
        saved.append(shutil.copy(os.path.join(o['index_dir'], f), SAVE_TO))
for s in saved:
    print('%8.1f مگابایت  %s' % (os.path.getsize(s) / 1048576, s))
print('\nتمام شد. همین فایل‌ها را موتور به کار می‌برد.')